Load data from individual daily netcdf files, using tools by Justine Charrel, concatenate to single dataset and export to single netcdf file.

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

#import personnal tools
import sys
sys.path.append('../tools/')
from imports import *
from info import *
from tools_generic import *

# Load files

In [ ]:
ds = open_single_dataset(
    prefix = 'FLOWCAPT',
    data_folder_path="/bdd/AWACA/TRANSECT/awacasurf",
    site='d17',
    year = '2025',
    month = '05',
    day = '19',
    drop_spc_bins=True,
    resample_mean=False)
ds

In [ ]:
data_folder_path='/bdd/AWACA/TRANSECT/awacasurf'
site_list=["d17","d47","d85","dmc"]
#site_list=['d17']
#site_list=["d47","d85","dmc"]


In [ ]:
prefix='SURF_TABLE'

begin_year="2024"
begin_month="12"
begin_day="01"
end_year="2025"
end_month="03"
end_day="13"

# begin_year="2024"
# begin_month="12"
# begin_day="01"
# end_year="2026"
# end_month="06"
# end_day="30"
begin_date_string = begin_year + begin_month + begin_day
end_date_string = end_year + end_month + end_day

In [ ]:
# for prefix in sensors:
if False:
    datasets = open_and_concatenate_datasets(
        prefix=prefix,
        data_folder_path="/bdd/AWACA/TRANSECT/awacasurf",
        site_list=site_list,
        begin_year=begin_year,
        begin_month=begin_month,
        begin_day=begin_day,
        end_year=end_year,
        end_month=end_month,
        end_day=end_day,
        resample_mean = True,
        reindex=True
    )

    #convert SPC snowflux for g/cm^2/s to g/m^2/s
    if prefix == 'SPC' :
        for site in site_list:
            if datasets[site] is not None:
                if 'snowflux' in datasets[site]:
                    print(f"Unit updated for snowflux at {site}")
                    datasets[site]['snowflux'] = datasets[site]['snowflux'] * 10000
                    datasets[site]['snowflux'].attrs['units'] = 'g/m^2/s'

    #export to single netcdf
    for site in site_list:
        ds=datasets.get(site)
        if ds is not None:
            #create netcdf file
            file_path=f'../../data/{prefix}_{site}_{begin_date_string}_{end_date_string}.netcdf'
            # Remove the file if it already exists
            if os.path.exists(file_path):
                os.remove(file_path)
                print(f"Existing file {file_path} removed.")
            ds.to_netcdf(file_path)

In [ ]:
wind_beginning=True # handle initial period where wind data stored differently

if wind_beginning:
    prefix='SURF_TABLE'

    begin_year="2024"
    begin_month="12"
    begin_day="01"
    end_year="2025"
    end_month="03"
    end_day="13"
        
    datasets = open_and_concatenate_datasets(
        prefix=prefix,
        data_folder_path="/bdd/AWACA/TRANSECT/awacasurf",
        site_list=site_list,
        begin_year=begin_year,
        begin_month=begin_month,
        begin_day=begin_day,
        end_year=end_year,
        end_month=end_month,
        end_day=end_day,
        resample_mean = True,
        reindex=True
    )

    #keep only relevant var
    var_list=["wspd1_Avg","wspd2_Avg","wspd3_Avg", "wdir"]
    rename_dict = { "wspd1_Avg":"wspd1",
                    "wspd2_Avg":"wspd2",
                    "wspd3_Avg":"wspd3"
                    }
    rename_dict_DMC = { "wspd1_Avg":"wspd1",
                        "wspd2_Avg":"wspd2"
                    }
    for site in site_list:
        ds=datasets.get(site)
        datasets[site]=ds.drop_vars([v for v in ds.data_vars if v not in var_list])
        if site=="dmc":
            datasets[site]=datasets[site].rename(rename_dict_DMC)
        else:
            datasets[site]=datasets[site].rename(rename_dict)
        

    #export to single netcdf
    for site in site_list:
        ds=datasets.get(site)
        if ds is not None:
            #create netcdf file
            file_path=f'../../data/WIND_BEGINNING_{site}_{begin_date_string}_{end_date_string}.netcdf'
            # Remove the file if it already exists
            if os.path.exists(file_path):
                os.remove(file_path)
                print(f"Existing file {file_path} removed.")
            ds.to_netcdf(file_path)


In [ ]:
datasets.get("d17")

In [ ]:
datasets.get("d47")

In [ ]:
datasets.get("d85")

In [ ]:
datasets.get("dmc")

# Basic plots

In [ ]:
ds=datasets.get("d17")
# ds=alias1
vars=['wspd1', 'wspd2', 'wspd3']
vars=['snowflux']
for var in vars:
    ds[var].plot()

# Export to unique netcdf

In [ ]:
for site in site_list:
    ds=datasets.get(site)
    if ds is not None:
        #create netcdf file
        file_path=f'../../data/{prefix}_{site}_{begin_date_string}_{end_date_string}.netcdf'
        # Remove the file if it already exists
        if os.path.exists(file_path):
            os.remove(file_path)
            print(f"Existing file {file_path} removed.")
        ds.to_netcdf(file_path)

# MRR
Load aggregated monthly files from scratch. 
Convert to 10min, aggregate and reexport as monthly netcdf.

In [ ]:
#functions that didn't work nice
def create_monthly_resampled_files(
    site: str,
    start_date: str,
    end_date: str,
    input_dir: str | Path,
    output_dir: str | Path,
    freq: str = "10min",
    var_name: str = "Zea",
) -> list[Path]:
    """Loop through months from start_date to end_date, resample the specified variable

    linearly, save individual NetCDF files, and return a list of created file
    paths.
    """
    input_dir = Path(input_dir)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    monthly_dates = pd.date_range(start=start_date, end=end_date, freq="MS")
    generated_files = []

    print(
        f"--- Starting monthly processing for site '{site}' ({start_date} to {end_date}) ---"
    )

    for start_of_month in monthly_dates:
        end_of_month = start_of_month + pd.offsets.MonthEnd(1)

        start_str = start_of_month.strftime("%Y%m%d")
        end_str = end_of_month.strftime("%Y%m%d")

        input_filename = f"mrr_{site}_{start_str}_{end_str}.nc"
        input_path = input_dir / input_filename

        output_filename = f"zea_averaged{freq}_mrr_{site}_{start_str}_{end_str}.nc"
        output_path = output_dir / output_filename

        if not input_path.exists():
            print(f"[SKIP] Missing input file: {input_path}")
            continue

        print(f"[PROCESSING] {input_filename} -> {output_filename}")

        # Core linear resampling logic
        with xr.open_dataset(input_path) as ds:
            da = ds[var_name]

            # Convert logarithmic dBZ to linear domain
            da_linear = 10 ** (da / 10)

            # Resample linearly and calculate mean
            da_lin_resampled = da_linear.resample(time=freq).mean(dim="time")

            # Convert back to dBZ
            da_dbz_resampled = 10 * np.log10(da_lin_resampled)
            da_dbz_resampled.name = var_name

            # Preserve attributes
            da_dbz_resampled.attrs = da.attrs
            da_dbz_resampled.attrs["processing_note"] = (
                f"Linearly averaged over {freq} intervals"
            )

            # Save resampled dataset
            da_dbz_resampled.to_netcdf(output_path)

        generated_files.append(output_path)

    print(f"--- Finished processing {len(generated_files)} files. ---\n")
    return generated_files

def aggregate_monthly_files(
    file_list: list[Path] | None = None,
    search_dir: str | Path | None = None,
    site: str = "d17",
    start_date: str = "2025-01-01",
    end_date: str = "2025-12-31",
    output_dir: str | Path = "../../data/MRR_aggregated",
    output_filename: str | None = None,
) -> Path | None:
    """Combines multiple monthly NetCDF files into a single continuous file.

    Accepts either an explicit list of file paths or searches inside `search_dir`.
    """
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    # Resolve files to aggregate
    files_to_combine = []
    if file_list:
        files_to_combine = [Path(f) for f in file_list if Path(f).exists()]
    elif search_dir:
        files_to_combine = sorted(list(Path(search_dir).glob(f"*{site}*.nc")))

    if not files_to_combine:
        print("[ERROR] No valid NetCDF files found to aggregate.")
        return None

    # Construct default output filename if not provided
    if output_filename is None:
        start_yyyymm = pd.to_datetime(start_date).strftime("%Y%m")
        end_yyyymm = pd.to_datetime(end_date).strftime("%Y%m")
        output_filename = f"zea_averaged_mrr_{site}_{start_yyyymm}_{end_yyyymm}_aggregated.nc"

    final_output_path = output_dir / output_filename

    print(f"--- Aggregating {len(files_to_combine)} monthly files ---")
    for f in files_to_combine:
        print(f"  + {f.name}")

    # Combine files along time dimension
    with xr.open_mfdataset(
        files_to_combine, combine="by_coords"
    ) as combined_ds:
        combined_ds.to_netcdf(final_output_path)

    print(f"[COMPLETE] Combined dataset written to: {final_output_path}\n")
    return final_output_path

In [ ]:

# Directories
# RAW_INPUT_DIR = Path("/scratchu/ptiengou/MRR_AWACA_formatting")
RAW_INPUT_DIR = Path("/data/ptiengou/AWACA/data/MRR_aggregated")
MONTHLY_OUT_DIR = Path("../../data/MRR_Zea_averaged/monthly")
FINAL_OUT_DIR = Path("../../data/MRR_Zea_averaged")

# Site d17 Input Files
input_files_d17 = [
    # f"{RAW_INPUT_DIR}/d17/mrr_d17_20250101_20250131.nc",
    f"{RAW_INPUT_DIR}/d17/mrr_d17_20250201_20250228.nc",
    f"{RAW_INPUT_DIR}/d17/mrr_d17_20250301_20250331.nc",
    f"{RAW_INPUT_DIR}/d17/mrr_d17_20250401_20250430.nc",
    f"{RAW_INPUT_DIR}/d17/mrr_d17_20250501_20250531.nc",
    # f"{RAW_INPUT_DIR}/d17/mrr_d17_20250601_20250630.nc",
    # f"{RAW_INPUT_DIR}/d17/mrr_d17_20250701_20250731.nc",
    # f"{RAW_INPUT_DIR}/d17/mrr_d17_20250801_20250831.nc",
    # f"{RAW_INPUT_DIR}/d17/mrr_d17_20250901_20250930.nc",
    # f"{RAW_INPUT_DIR}/d17/mrr_d17_20251001_20251031.nc",
    f"{RAW_INPUT_DIR}/d17/mrr_d17_20251101_20251130.nc",
    f"{RAW_INPUT_DIR}/d17/mrr_d17_20251201_20251231.nc",
]

# Site d47 Input Files
input_files_d47 = [
    f"{RAW_INPUT_DIR}/d47/mrr_d47_20250101_20250131.nc",
    f"{RAW_INPUT_DIR}/d47/mrr_d47_20250201_20250228.nc",
    f"{RAW_INPUT_DIR}/d47/mrr_d47_20250301_20250331.nc",
    f"{RAW_INPUT_DIR}/d47/mrr_d47_20250401_20250430.nc",
    f"{RAW_INPUT_DIR}/d47/mrr_d47_20250501_20250531.nc",
    f"{RAW_INPUT_DIR}/d47/mrr_d47_20250601_20250630.nc",
    f"{RAW_INPUT_DIR}/d47/mrr_d47_20250701_20250731.nc",
    f"{RAW_INPUT_DIR}/d47/mrr_d47_20250801_20250831.nc",
    f"{RAW_INPUT_DIR}/d47/mrr_d47_20250901_20250930.nc",
    f"{RAW_INPUT_DIR}/d47/mrr_d47_20251001_20251031.nc",
    f"{RAW_INPUT_DIR}/d47/mrr_d47_20251101_20251130.nc",
    f"{RAW_INPUT_DIR}/d47/mrr_d47_20251201_20251231.nc",
]
    
# Site d17 Resampled Monthly Outputs
output_files_d17 = [
    # f"{MONTHLY_OUT_DIR}/zea_averaged10min_mrr_d17_20250101_20250131.nc",
    f"{MONTHLY_OUT_DIR}/zea_averaged10min_mrr_d17_20250201_20250228.nc",
    f"{MONTHLY_OUT_DIR}/zea_averaged10min_mrr_d17_20250301_20250331.nc",
    f"{MONTHLY_OUT_DIR}/zea_averaged10min_mrr_d17_20250401_20250430.nc",
    f"{MONTHLY_OUT_DIR}/zea_averaged10min_mrr_d17_20250501_20250531.nc",
    # f"{MONTHLY_OUT_DIR}/zea_averaged10min_mrr_d17_20250601_20250630.nc",
    # f"{MONTHLY_OUT_DIR}/zea_averaged10min_mrr_d17_20250701_20250731.nc",
    # f"{MONTHLY_OUT_DIR}/zea_averaged10min_mrr_d17_20250801_20250831.nc",
    # f"{MONTHLY_OUT_DIR}/zea_averaged10min_mrr_d17_20250901_20250930.nc",
    # f"{MONTHLY_OUT_DIR}/zea_averaged10min_mrr_d17_20251001_20251031.nc",
    f"{MONTHLY_OUT_DIR}/zea_averaged10min_mrr_d17_20251101_20251130.nc",
    f"{MONTHLY_OUT_DIR}/zea_averaged10min_mrr_d17_20251201_20251231.nc",
]

# Site d47 Resampled Monthly Outputs
output_files_d47 = [
    f"{MONTHLY_OUT_DIR}/zea_averaged10min_mrr_d47_20250101_20250131.nc",
    f"{MONTHLY_OUT_DIR}/zea_averaged10min_mrr_d47_20250201_20250228.nc",
    f"{MONTHLY_OUT_DIR}/zea_averaged10min_mrr_d47_20250301_20250331.nc",
    f"{MONTHLY_OUT_DIR}/zea_averaged10min_mrr_d47_20250401_20250430.nc",
    f"{MONTHLY_OUT_DIR}/zea_averaged10min_mrr_d47_20250501_20250531.nc",
    f"{MONTHLY_OUT_DIR}/zea_averaged10min_mrr_d47_20250601_20250630.nc",
    f"{MONTHLY_OUT_DIR}/zea_averaged10min_mrr_d47_20250701_20250731.nc",
    f"{MONTHLY_OUT_DIR}/zea_averaged10min_mrr_d47_20250801_20250831.nc",
    f"{MONTHLY_OUT_DIR}/zea_averaged10min_mrr_d47_20250901_20250930.nc",
    f"{MONTHLY_OUT_DIR}/zea_averaged10min_mrr_d47_20251001_20251031.nc",
    f"{MONTHLY_OUT_DIR}/zea_averaged10min_mrr_d47_20251101_20251130.nc",
    f"{MONTHLY_OUT_DIR}/zea_averaged10min_mrr_d47_20251201_20251231.nc",
]

In [ ]:
i=0
for filename in input_files_d17:
    print(f'Opening {filename}')
    output_filename = output_files_d17[i]
    mrr = xr.open_dataset(filename)
    print('Resampling to 10min')
    zea = mrr['Zea']
    zea_linear = 10 ** (zea / 10)
    zea_lin_10mn = zea_linear.resample(time='10min').mean(dim='time')
    print('Resampling complete')
    zea_dbz_10mn = 10 * np.log10(zea_lin_10mn)
    print(f'Writing netcdf output file : {output_filename}')
    zea_dbz_10mn.to_netcdf(output_filename)

    i+=1

In [ ]:
i=0
for filename in input_files_d47:
    print(f'Opening {filename}')
    output_filename = output_files_d47[i]
    mrr = xr.open_dataset(filename)
    print('Extracting Zea')
    zea = mrr['Zea']
    print('Resampling to 10min')
    zea_linear = 10 ** (zea / 10)
    zea_lin_10mn = zea_linear.resample(time='10min').mean(dim='time')
    print('Resampling complete')
    zea_dbz_10mn = 10 * np.log10(zea_lin_10mn)
    print(f'Writing netcdf output file : {output_filename}')
    zea_dbz_10mn.to_netcdf(output_filename)

    i+=1

In [ ]:
# Step 2: Combine all created monthly files into one
if created_files:
    aggregate_monthly_files(
        file_list=created_files,
        site=SITE,
        start_date=START,
        end_date=END,
        output_dir=FINAL_OUT_DIR,
    )